In [2]:
#!pip install ddgs trafilatura 

### Setup

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from pprint import pprint
import json
from ddgs import DDGS
import trafilatura

load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("API_KEY is missing.")

### Step 1: Define the Tools

In [2]:
def search_web(query):
    ddgs = DDGS()
    results = ddgs.text(query, max_results=3)
    print(f" \u2705  Got results")
    return json.dumps(results, indent=2)

search_web("AI in healthcare in 2030")

 ✅  Got results


'[\n  {\n    "title": "7 ways AI is transforming healthcare | World Economic Forum",\n    "href": "https://www.weforum.org/stories/2025/08/ai-transforming-global-health/",\n    "body": "With 4.5 billion people currently without access to essential healthcare services and a health worker shortage of 11 million expected by 2030, AI has the potential to help bridge that gap and revolutionize global healthcare."\n  },\n  {\n    "title": "AI in Healthcare: 3 Revolutionary Trends by 2030 | Kindo",\n    "href": "https://www.kindo.ai/blog/how-ai-will-change-healthcare-by-2030",\n    "body": "AI transforms healthcare via predictive analytics, precision medicine, and automated diagnostics. Explore 3 key shifts defining the 2030 medical landscape."\n  },\n  {\n    "title": "Artificial intelligence in healthcare: transforming the practice of medicine - PMC",\n    "href": "https://pmc.ncbi.nlm.nih.gov/articles/PMC8285156/",\n    "body": "In this review article, we outline recent breakthroughs in th

In [3]:
def fetch_url(url: str):
    downloaded = trafilatura.fetch_url(url)
    if downloaded:
        print(f" \u2705  Successfully downloaded content from {url} and it's length is {len(downloaded)} characters")
        return trafilatura.extract(downloaded)
    else:
        print(f"Failed to download content from {url}")
        return None

fetch_url("https://en.wikipedia.org/wiki/Artificial_intelligence") 

 ✅  Successfully downloaded content from https://en.wikipedia.org/wiki/Artificial_intelligence and it's length is 1250768 characters


'Artificial intelligence\n| Part of a series on |\n| Artificial intelligence (AI) |\n|---|\nArtificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximize their chances of achieving defined goals.[1]\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, and play and analysis in strategy games (e.g., chess and Go). Since the 2020s, generative AI has become widely available to generate images, audio, and videos from text prompts.\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural la

### Step 2: Describe as LLM Tools

In [4]:
tools = []

In [5]:
search_web_function = {
    "name": "search_web",
    "description": "Search the web using duckduckgo search engine and return relevant results.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The search query to find relevant information on the web."
            }
        },
        "required": ["query"]
    }
}

tools.append({"type": "function", "function": search_web_function})

In [6]:
fetch_url_function = {
    "name": "fetch_url",
    "description": "Fetch the content of a URL and return it.",
    "parameters": {
        "type": "object",
        "properties": {
            "url": {
                "type": "string",
                "description": "The URL to fetch content from."
            }
        },
        "required": ["url"]
    }
}

tools.append({"type": "function", "function": fetch_url_function})

In [7]:
tools

[{'type': 'function',
  'function': {'name': 'search_web',
   'description': 'Search the web using duckduckgo search engine and return relevant results.',
   'parameters': {'type': 'object',
    'properties': {'query': {'type': 'string',
      'description': 'The search query to find relevant information on the web.'}},
    'required': ['query']}}},
 {'type': 'function',
  'function': {'name': 'fetch_url',
   'description': 'Fetch the content of a URL and return it.',
   'parameters': {'type': 'object',
    'properties': {'url': {'type': 'string',
      'description': 'The URL to fetch content from.'}},
    'required': ['url']}}}]

In [8]:
def handle_tool_call(tool_calls):
    tool_results = []
    
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        
        if function_name == "search_web":
            result = search_web(args["query"])
            content = f"Search result: {result}"
        elif function_name == "fetch_url":
            result = fetch_url(args["url"])
            content = f"Fetched URL: {result}"
        else:
            content = f"Unknown tool: {function_name}"
        
        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }
        
        tool_results.append(tool_call_result)
        
    return tool_results

In [9]:
RESEARCH_AGENT_PROMPT = """You are a research assistant. You have access to the following tools: search_web and fetch_url.
Use search_web to find relevant information on the web and fetch_url to get the content of a specific URL. Always try to use these tools to get accurate and up-to-date information. If you need to find information, use search_web. If you find a relevant URL, use fetch_url to get the content of that URL. Always provide the most relevant and accurate information in your responses."""

### Step 3: Research Function

The `research` function orchestrates a multi-iteration research agent that:
1. Searches the web and fetches URLs to gather raw material
2. Drafts an initial `research_brief` using the collected content
3. Iteratively critiques and rewrites the brief up to `max_iterations` times
4. Returns the final polished brief as a string

**Arguments**
- `topic` – the subject to research
- `max_iterations` – how many critique-and-rewrite passes to run (default 3)
- `num_urls` – how many URLs to fetch during the research phase (default 5)

In [10]:
def research(topic: str, max_iterations: int = 3, num_urls: int = 5) -> str:
    """
    Run a multi-iteration research agent that produces a polished research_brief.

    Args:
        topic:          Subject to research.
        max_iterations: Number of critique-and-rewrite cycles after the initial draft.
        num_urls:       Maximum number of URLs to fetch during the research phase.

    Returns:
        A markdown-formatted research brief as a string.
    """
    client = OpenAI()

    # ------------------------------------------------------------------
    # Phase 1 – Gather raw research material
    # ------------------------------------------------------------------
    print(f"\n{'='*60}")
    print(f"RESEARCHING: {topic}")
    print(f"max_iterations={max_iterations}  num_urls={num_urls}")
    print(f"{'='*60}\n")

    gathered_content: list[str] = []
    urls_fetched = 0

    gather_messages = [
        {"role": "system", "content": (
            "You are a meticulous research assistant. "
            "Your job right now is ONLY to gather raw information – do NOT write a report yet. "
            f"Search for information about the topic and fetch up to {num_urls} relevant URLs. "
            "After you have gathered enough material, reply with exactly: RESEARCH_COMPLETE"
        )},
        {"role": "user", "content": f"Gather comprehensive research material on: {topic}"}
    ]

    # Tool-call loop for gathering
    while urls_fetched < num_urls:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=gather_messages,
            tools=tools,
            tool_choice="auto"
        )
        msg = response.choices[0].message
        gather_messages.append(msg)

        # No tool call → model is done gathering
        if not msg.tool_calls:
            print("[Gather] Agent signalled completion.")
            break

        tool_results = handle_tool_call(msg.tool_calls)

        for tr, tc in zip(tool_results, msg.tool_calls):
            if tc.function.name == "fetch_url":
                urls_fetched += 1
                gathered_content.append(tr["content"])
            elif tc.function.name == "search_web":
                gathered_content.append(tr["content"])

        gather_messages.extend(tool_results)

        if urls_fetched >= num_urls:
            print(f"[Gather] Reached target of {num_urls} URLs.")
            break

    raw_material = "\n\n".join(gathered_content)
    print(f"\n[Gather] Collected {len(raw_material):,} characters of raw material.\n")

    # ------------------------------------------------------------------
    # Phase 2 – Draft the initial research_brief
    # ------------------------------------------------------------------
    print("[Draft] Writing initial research brief...")

    draft_prompt = (
        f"Using the research material below, write a comprehensive, well-structured "
        f"research brief on the topic: '{topic}'.\n\n"
        "Format the brief in Markdown with the following sections:\n"
        "1. Executive Summary\n"
        "2. Background & Context\n"
        "3. Key Findings\n"
        "4. Analysis & Implications\n"
        "5. Conclusion\n"
        "6. Sources (list URLs referenced)\n\n"
        f"--- RAW RESEARCH MATERIAL ---\n{raw_material[:12000]}"
    )

    draft_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": RESEARCH_AGENT_PROMPT},
            {"role": "user",   "content": draft_prompt}
        ]
    )
    research_brief = draft_response.choices[0].message.content
    print("[Draft] Initial brief written.\n")

    # ------------------------------------------------------------------
    # Phase 3 – Iterative critique-and-rewrite loop
    # ------------------------------------------------------------------
    for iteration in range(1, max_iterations + 1):
        print(f"[Iteration {iteration}/{max_iterations}] Critiquing brief...")

        critique_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": (
                    "You are a rigorous editor. Critique the research brief below. "
                    "Identify specific weaknesses: gaps in coverage, unclear arguments, "
                    "unsupported claims, poor structure, or missing context. "
                    "Be concise and actionable."
                )},
                {"role": "user", "content": (
                    f"Topic: {topic}\n\nBrief to critique:\n{research_brief}"
                )}
            ]
        )
        critique = critique_response.choices[0].message.content
        print(f"[Iteration {iteration}] Critique received. Rewriting...")

        rewrite_response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": RESEARCH_AGENT_PROMPT},
                {"role": "user", "content": (
                    f"Rewrite and improve the following research brief on '{topic}' "
                    f"based on this critique:\n\n"
                    f"--- CRITIQUE ---\n{critique}\n\n"
                    f"--- CURRENT BRIEF ---\n{research_brief}\n\n"
                    "Return only the improved brief in Markdown. Keep all six sections."
                )}
            ]
        )
        research_brief = rewrite_response.choices[0].message.content
        print(f"[Iteration {iteration}] Brief updated.\n")

    print(f"{'='*60}")
    print("RESEARCH COMPLETE")
    print(f"{'='*60}\n")
    return research_brief


### Step 4: Run the Research Agent

In [11]:
# Example usage
brief = research(
    topic="The impact of large language models on software engineering workflows",
    max_iterations=2,
    num_urls=4
)

print(brief)



RESEARCHING: The impact of large language models on software engineering workflows
max_iterations=2  num_urls=4

 ✅  Got results
 ✅  Successfully downloaded content from https://arxiv.org/html/2308.10620v6 and it's length is 1306066 characters
 ✅  Successfully downloaded content from https://arxiv.org/abs/2308.10620 and it's length is 47402 characters
Failed to download content from https://ieeexplore.ieee.org/document/10109345/
 ✅  Successfully downloaded content from https://arxiv.org/abs/2308.10620 and it's length is 47402 characters
[Gather] Reached target of 4 URLs.

[Gather] Collected 367,574 characters of raw material.

[Draft] Writing initial research brief...
[Draft] Initial brief written.

[Iteration 1/2] Critiquing brief...
[Iteration 1] Critique received. Rewriting...
[Iteration 1] Brief updated.

[Iteration 2/2] Critiquing brief...
[Iteration 2] Critique received. Rewriting...
[Iteration 2] Brief updated.

RESEARCH COMPLETE

# The Impact of Large Language Models on Softwa

In [12]:
# Example usage
brief1 = research(
    topic="The impact of large language models on software engineering workflows",
    max_iterations=3,
    num_urls=3
)

pprint(brief1)


RESEARCHING: The impact of large language models on software engineering workflows
max_iterations=3  num_urls=3

 ✅  Got results
Failed to download content from https://dl.acm.org/doi/10.1145/3773084
Failed to download content from https://link.springer.com/article/10.1007/s11432-025-4670-0
Failed to download content from https://cs.ccsu.edu/~stan/classes/CS498/readings/06-Application+of+Large+Language+Models+to+Software+Engineering.pdf
[Gather] Reached target of 3 URLs.

[Gather] Collected 1,628 characters of raw material.

[Draft] Writing initial research brief...
[Draft] Initial brief written.

[Iteration 1/3] Critiquing brief...
[Iteration 1] Critique received. Rewriting...
[Iteration 1] Brief updated.

[Iteration 2/3] Critiquing brief...
[Iteration 2] Critique received. Rewriting...
[Iteration 2] Brief updated.

[Iteration 3/3] Critiquing brief...
[Iteration 3] Critique received. Rewriting...
[Iteration 3] Brief updated.

RESEARCH COMPLETE

('# The Impact of Large Language Models